# 05 — Model Optimization

This notebook investigates additional optimization experiments for the BuySense purchase-prediction models. The analysis focuses on engineered features, PageValues sensitivity, and other evidence needed to support the final model configuration.

The optimization experiments use the same train-test split, feature-engineering rules, preprocessing strategy, and 5-fold stratified cross-validation defined during model development.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    GridSearchCV
)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    make_scorer
)

import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv("../data/raw/online_shoppers_intention.csv")

print("Dataset shape:", df.shape)

df.head()
X = df.drop(columns=["Revenue"])
y = df["Revenue"].astype(int)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nClass distribution:")
print(y.value_counts())

print("\nClass proportions:")
print(y.value_counts(normalize=True))

Dataset shape: (12330, 18)
X shape: (12330, 17)
y shape: (12330,)

Class distribution:
Revenue
0    10422
1     1908
Name: count, dtype: int64

Class proportions:
Revenue
0    0.845255
1    0.154745
Name: proportion, dtype: float64


In [3]:
X_train_original, X_test_original, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train_original.shape)
print("Test shape:", X_test_original.shape)

Training shape: (9864, 17)
Test shape: (2466, 17)


In [4]:
def add_engineered_features(data):
    data = data.copy()

    data["TotalPages"] = (
        data["Administrative"]
        + data["Informational"]
        + data["ProductRelated"]
    )

    data["TotalDuration"] = (
        data["Administrative_Duration"]
        + data["Informational_Duration"]
        + data["ProductRelated_Duration"]
    )

    data["AvgDurationPerPage"] = np.where(
        data["TotalPages"] > 0,
        data["TotalDuration"] / data["TotalPages"],
        0
    )

    data["ProductPageShare"] = np.where(
        data["TotalPages"] > 0,
        data["ProductRelated"] / data["TotalPages"],
        0
    )

    data["ProductDurationShare"] = np.where(
        data["TotalDuration"] > 0,
        data["ProductRelated_Duration"]
        / data["TotalDuration"],
        0
    )

    return data

In [5]:
X_train_engineered = add_engineered_features(
    X_train_original
)

X_test_engineered = add_engineered_features(
    X_test_original
)

engineered_features = [
    "TotalPages",
    "TotalDuration",
    "AvgDurationPerPage",
    "ProductPageShare",
    "ProductDurationShare"
]

print("Engineered features:")
print(engineered_features)

print(
    "\nOriginal training shape:",
    X_train_original.shape
)

print(
    "Engineered training shape:",
    X_train_engineered.shape
)

Engineered features:
['TotalPages', 'TotalDuration', 'AvgDurationPerPage', 'ProductPageShare', 'ProductDurationShare']

Original training shape: (9864, 17)
Engineered training shape: (9864, 22)


In [6]:
categorical_features = [
    "Month",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "VisitorType"
]

boolean_features = [
    "Weekend"
]

numeric_features_original = [
    col
    for col in X_train_original.columns
    if col not in categorical_features + boolean_features
]

numeric_features_engineered = [
    col
    for col in X_train_engineered.columns
    if col not in categorical_features + boolean_features
]

print(
    "Number of original numeric features:",
    len(numeric_features_original)
)

print(
    "Number of engineered numeric features:",
    len(numeric_features_engineered)
)

Number of original numeric features: 10
Number of engineered numeric features: 15


In [7]:
preprocessor_tree_original = ColumnTransformer(
    transformers=[
        (
            "num",
            "passthrough",
            numeric_features_original
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        ),
        (
            "bool",
            "passthrough",
            boolean_features
        )
    ]
)

preprocessor_tree_engineered = ColumnTransformer(
    transformers=[
        (
            "num",
            "passthrough",
            numeric_features_engineered
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        ),
        (
            "bool",
            "passthrough",
            boolean_features
        )
    ]
)

In [8]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(
        precision_score,
        zero_division=0
    ),
    "recall": make_scorer(
        recall_score,
        zero_division=0
    ),
    "f1": make_scorer(
        f1_score,
        zero_division=0
    ),
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision"
}

In [9]:
def summarize_cv(
    model_name,
    pipeline,
    X_data,
    y_data
):
    scores = cross_validate(
        pipeline,
        X_data,
        y_data,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        n_jobs=-1
    )

    row = {
        "Model": model_name
    }

    for metric in scoring:
        row[f"CV_{metric}_mean"] = (
            scores[f"test_{metric}"].mean()
        )

        row[f"CV_{metric}_std"] = (
            scores[f"test_{metric}"].std()
        )

    row["Train_F1_mean"] = (
        scores["train_f1"].mean()
    )

    row["Validation_F1_mean"] = (
        scores["test_f1"].mean()
    )

    return row, scores

Random Forest: Original Features

In [10]:
rf_original = Pipeline([
    (
        "preprocessor",
        preprocessor_tree_original
    ),
    (
        "model",
        RandomForestClassifier(
            random_state=42,
            n_jobs=-1
        )
    )
])

Random Forest: Engineered Features

In [11]:
rf_engineered = Pipeline([
    (
        "preprocessor",
        preprocessor_tree_engineered
    ),
    (
        "model",
        RandomForestClassifier(
            random_state=42,
            n_jobs=-1
        )
    )
])

Original Features

In [12]:
rf_original_row, rf_original_scores = summarize_cv(
    "Random Forest - Original Features",
    rf_original,
    X_train_original,
    y_train
)

Engineered Features

In [13]:
rf_engineered_row, rf_engineered_scores = summarize_cv(
    "Random Forest - Engineered Features",
    rf_engineered,
    X_train_engineered,
    y_train
)

In [14]:
feature_comparison = pd.DataFrame([
    rf_original_row,
    rf_engineered_row
])

feature_comparison[
    [
        "Model",
        "CV_accuracy_mean",
        "CV_precision_mean",
        "CV_recall_mean",
        "CV_f1_mean",
        "CV_roc_auc_mean",
        "CV_pr_auc_mean"
    ]
]

,Model,CV_accuracy_mean,CV_precision_mean,CV_recall_mean,CV_f1_mean,CV_roc_auc_mean,CV_pr_auc_mean
0,Random Forest - Original Features,0.899332,0.753262,0.518991,0.614473,0.923584,0.732332
1,Random Forest - Engineered Features,0.900143,0.767610,0.507845,0.611127,0.921270,0.726216


PageValues sensitivity experiment

In [15]:
numeric_features_without_pagevalues = [
    col
    for col in numeric_features_engineered
    if col != "PageValues"
]

print("Number of numeric features with PageValues:",
      len(numeric_features_engineered))

print("Number of numeric features without PageValues:",
      len(numeric_features_without_pagevalues))

Number of numeric features with PageValues: 15
Number of numeric features without PageValues: 14


In [16]:
preprocessor_tree_without_pagevalues = ColumnTransformer(
    transformers=[
        (
            "num",
            "passthrough",
            numeric_features_without_pagevalues
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        ),
        (
            "bool",
            "passthrough",
            boolean_features
        )
    ]
)

Random Forest without PageValues

In [17]:
rf_without_pagevalues = Pipeline([
    (
        "preprocessor",
        preprocessor_tree_without_pagevalues
    ),
    (
        "model",
        RandomForestClassifier(
            random_state=42,
            n_jobs=-1
        )
    )
])

In [18]:
rf_without_pagevalues_row, rf_without_pagevalues_scores = summarize_cv(
    "Random Forest - Without PageValues",
    rf_without_pagevalues,
    X_train_engineered,
    y_train
)

In [19]:
pagevalues_comparison = pd.DataFrame([
    rf_engineered_row,
    rf_without_pagevalues_row
])

pagevalues_comparison[
    [
        "Model",
        "CV_accuracy_mean",
        "CV_precision_mean",
        "CV_recall_mean",
        "CV_f1_mean",
        "CV_roc_auc_mean",
        "CV_pr_auc_mean"
    ]
]

,Model,CV_accuracy_mean,CV_precision_mean,CV_recall_mean,CV_f1_mean,CV_roc_auc_mean,CV_pr_auc_mean
0,Random Forest - Engineered Features,0.900143,0.767610,0.507845,0.611127,0.921270,0.726216
1,Random Forest - Without PageValues,0.845296,0.507221,0.072082,0.125764,0.771429,0.360519


In [20]:
optimization_summary = pd.DataFrame([
    rf_original_row,
    rf_engineered_row,
    rf_without_pagevalues_row
])

optimization_summary[
    [
        "Model",
        "CV_accuracy_mean",
        "CV_precision_mean",
        "CV_recall_mean",
        "CV_f1_mean",
        "CV_roc_auc_mean",
        "CV_pr_auc_mean"
    ]
].sort_values(
    "CV_pr_auc_mean",
    ascending=False
)

,Model,CV_accuracy_mean,CV_precision_mean,CV_recall_mean,CV_f1_mean,CV_roc_auc_mean,CV_pr_auc_mean
0,Random Forest - Original Features,0.899332,0.753262,0.518991,0.614473,0.923584,0.732332
1,Random Forest - Engineered Features,0.900143,0.767610,0.507845,0.611127,0.921270,0.726216
2,Random Forest - Without PageValues,0.845296,0.507221,0.072082,0.125764,0.771429,0.360519


### Optimization Experiment Summary

The optimization experiments investigated the contribution of engineered features and the `PageValues` variable to Random Forest performance.

The engineered-feature configuration produced a small increase in accuracy and precision compared with the original-feature configuration, but showed slightly lower recall, F1-score, ROC-AUC, and PR-AUC.

In contrast, removing `PageValues` resulted in a substantial decrease in predictive performance across all evaluated metrics. In particular, recall, F1-score, ROC-AUC, and PR-AUC decreased considerably. This indicates that `PageValues` provides substantial predictive information for the purchase-prediction task in this dataset.

Therefore, `PageValues` was retained for the subsequent modelling analysis.

### Cross-Validation Stability Analysis

Five-fold stratified cross-validation was used to assess the stability of the Random Forest model with engineered features.

The model achieved a mean PR-AUC of 0.7262 (SD = 0.0198) and a mean ROC-AUC of 0.9213 (SD = 0.0047). The mean F1-score was 0.6111 (SD = 0.0288), while the mean recall and precision were 0.5078 (SD = 0.0293) and 0.7676 (SD = 0.0255), respectively.

The relatively small standard deviations indicate that the model's performance was reasonably consistent across the five validation folds. This provides evidence that the observed cross-validation performance was not strongly dependent on a particular validation split.

### Fold-Level Cross-Validation Results

The performance of the Random Forest model was evaluated separately across the five stratified validation folds. The fold-level results are presented below to examine variation in performance between different validation subsets.

In [21]:
rf_cv_results = pd.DataFrame({
    "Fold": range(1, 6),
    "PR_AUC": rf_engineered_scores["test_pr_auc"],
    "ROC_AUC": rf_engineered_scores["test_roc_auc"],
    "F1": rf_engineered_scores["test_f1"],
    "Recall": rf_engineered_scores["test_recall"],
    "Precision": rf_engineered_scores["test_precision"]
})

rf_cv_results

,Fold,PR_AUC,ROC_AUC,F1,Recall,Precision
0,1,0.730421,0.920429,0.598826,0.501639,0.742718
1,2,0.703108,0.916522,0.582834,0.478689,0.744898
2,3,0.703681,0.918054,0.583501,0.475410,0.755208
3,4,0.751157,0.930127,0.638132,0.535948,0.788462
4,5,0.742711,0.921216,0.652344,0.547541,0.806763


In [22]:
rf_cv_summary = pd.DataFrame({
    "Metric": [
        "PR-AUC",
        "ROC-AUC",
        "F1",
        "Recall",
        "Precision"
    ],
    "Mean": [
        rf_engineered_scores["test_pr_auc"].mean(),
        rf_engineered_scores["test_roc_auc"].mean(),
        rf_engineered_scores["test_f1"].mean(),
        rf_engineered_scores["test_recall"].mean(),
        rf_engineered_scores["test_precision"].mean()
    ],
    "Std": [
        rf_engineered_scores["test_pr_auc"].std(),
        rf_engineered_scores["test_roc_auc"].std(),
        rf_engineered_scores["test_f1"].std(),
        rf_engineered_scores["test_recall"].std(),
        rf_engineered_scores["test_precision"].std()
    ]
})

rf_cv_summary

,Metric,Mean,Std
0,PR-AUC,0.726216,0.019767
1,ROC-AUC,0.921270,0.004734
2,F1,0.611127,0.028786
3,Recall,0.507845,0.029347
4,Precision,0.767610,0.025523


## Final Model Comparison

The tuned models were evaluated on the held-out test set using accuracy, precision, recall, F1-score, ROC-AUC, and PR-AUC. Since the Purchase class represents a smaller proportion of observations, PR-AUC is considered alongside the other evaluation metrics.

In [23]:
test_results_df = pd.DataFrame({
    "Model": [
        "Logistic Regression - Tuned",
        "KNN - Tuned",
        "Decision Tree - Tuned",
        "Random Forest - Tuned"
    ],
    "Test_Accuracy": [
        0.872263,
        0.874290,
        0.899027,
        0.894972
    ],
    "Test_Precision": [
        0.568507,
        0.743243,
        0.711111,
        0.753086
    ],
    "Test_Recall": [
        0.727749,
        0.287958,
        0.586387,
        0.479058
    ],
    "Test_F1": [
        0.638347,
        0.415094,
        0.642755,
        0.585600
    ],
    "Test_ROC_AUC": [
        0.896519,
        0.848391,
        0.917496,
        0.918186
    ],
    "Test_PR_AUC": [
        0.629326,
        0.581001,
        0.685826,
        0.728636
    ]
})

test_results_df

,Model,Test_Accuracy,Test_Precision,Test_Recall,Test_F1,Test_ROC_AUC,Test_PR_AUC
0,Logistic Regression - Tuned,0.872263,0.568507,0.727749,0.638347,0.896519,0.629326
1,KNN - Tuned,0.874290,0.743243,0.287958,0.415094,0.848391,0.581001
2,Decision Tree - Tuned,0.899027,0.711111,0.586387,0.642755,0.917496,0.685826
3,Random Forest - Tuned,0.894972,0.753086,0.479058,0.585600,0.918186,0.728636


In [24]:
final_model_comparison = test_results_df[
    [
        "Model",
        "Test_Accuracy",
        "Test_Precision",
        "Test_Recall",
        "Test_F1",
        "Test_ROC_AUC",
        "Test_PR_AUC"
    ]
].copy()

final_model_comparison.sort_values(
    "Test_PR_AUC",
    ascending=False
)

,Model,Test_Accuracy,Test_Precision,Test_Recall,Test_F1,Test_ROC_AUC,Test_PR_AUC
3,Random Forest - Tuned,0.894972,0.753086,0.479058,0.585600,0.918186,0.728636
2,Decision Tree - Tuned,0.899027,0.711111,0.586387,0.642755,0.917496,0.685826
0,Logistic Regression - Tuned,0.872263,0.568507,0.727749,0.638347,0.896519,0.629326
1,KNN - Tuned,0.874290,0.743243,0.287958,0.415094,0.848391,0.581001


## Final Model Evaluation

The tuned models were evaluated using both cross-validation and a held-out test set. The results show that the models exhibit different strengths across the evaluation metrics.

Random Forest achieved the highest cross-validation PR-AUC (0.7493) and the highest test PR-AUC (0.7286) among the evaluated tuned models. Decision Tree achieved the highest test accuracy (0.8990), while Logistic Regression achieved the highest test recall (0.7277).

These results demonstrate that model performance depends on the selected evaluation metric. Therefore, accuracy alone should not be used to evaluate the BuySense purchase-prediction problem, particularly because the Purchase class is less common than the No Purchase class.

In [25]:
cv_pr_auc_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression - Tuned",
        "KNN - Tuned",
        "Decision Tree - Tuned",
        "Random Forest - Tuned"
    ],
    "CV_PR_AUC": [
        0.666409,
        0.623967,
        0.701188,
        0.749287
    ]
})

cv_pr_auc_comparison

,Model,CV_PR_AUC
0,Logistic Regression - Tuned,0.666409
1,KNN - Tuned,0.623967
2,Decision Tree - Tuned,0.701188
3,Random Forest - Tuned,0.749287


In [26]:
final_comparison = final_model_comparison.merge(
    cv_pr_auc_comparison,
    on="Model",
    how="left"
)

final_comparison = final_comparison[
    [
        "Model",
        "CV_PR_AUC",
        "Test_PR_AUC",
        "Test_Accuracy",
        "Test_Precision",
        "Test_Recall",
        "Test_F1",
        "Test_ROC_AUC"
    ]
]

final_comparison

,Model,CV_PR_AUC,Test_PR_AUC,Test_Accuracy,Test_Precision,Test_Recall,Test_F1,Test_ROC_AUC
0,Logistic Regression - Tuned,0.666409,0.629326,0.872263,0.568507,0.727749,0.638347,0.896519
1,KNN - Tuned,0.623967,0.581001,0.874290,0.743243,0.287958,0.415094,0.848391
2,Decision Tree - Tuned,0.701188,0.685826,0.899027,0.711111,0.586387,0.642755,0.917496
3,Random Forest - Tuned,0.749287,0.728636,0.894972,0.753086,0.479058,0.585600,0.918186


## Optimization Conclusions

The optimization experiments examined the effect of feature engineering and the `PageValues` variable on Random Forest performance.

The engineered-feature Random Forest produced a small change in performance compared with the original-feature model. Removing `PageValues`, however, resulted in a substantial reduction in PR-AUC, ROC-AUC, recall, and F1-score, indicating that `PageValues` provides important predictive information for this dataset.

The five-fold cross-validation analysis also showed relatively small standard deviations across the evaluated metrics, indicating reasonably consistent performance across validation folds.

Finally, comparison of the tuned models showed different strengths across the evaluation metrics. Random Forest obtained the highest PR-AUC on both cross-validation and the held-out test set, while Decision Tree achieved the highest test accuracy and Logistic Regression achieved the highest test recall.

Overall, the experiments demonstrate the importance of evaluating multiple metrics rather than relying on accuracy alone when assessing purchase prediction performance.